# Deep Learning / NLP Text Classifier: Spam vs. Not Spam Email

**Framework:** TensorFlow / Keras  
**Techniques:** Text cleaning, TF-IDF vectorization, neural network, dropout, batch normalization, early stopping, training curves, and predictions on unseen messages.

## Project objective
Build a binary text classifier that predicts whether an SMS/email message is **Spam** or **Not Spam (Ham)**.

> Run the notebook from top to bottom. The dataset is downloaded from a public repository, so an internet connection is required for the first data-loading step.

## 1. Import libraries

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 2. Load the dataset
The SMS Spam Collection contains labeled messages. Labels are `ham` (not spam) and `spam`.

In [ ]:
# Public SMS Spam Collection mirror (tab-separated: label, message)
DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

try:
    df = pd.read_csv(DATA_URL, sep="\t", header=None, names=["label", "text"])
except Exception as e:
    raise RuntimeError(
        "Could not download the dataset. Check your internet connection, then rerun this cell. "
        "You can also download an SMS Spam Collection TSV file and load it with "
        "pd.read_csv('your_file.tsv', sep='\\t', header=None, names=['label', 'text'])."
    ) from e

df = df.dropna(subset=["label", "text"]).copy()
df["label"] = df["label"].str.lower().str.strip()
df = df[df["label"].isin(["ham", "spam"])].drop_duplicates(subset=["text"])
df["target"] = (df["label"] == "spam").astype(int)

print("Dataset shape:", df.shape)
display(df.head())
print("\nClass counts:")
display(df["label"].value_counts())

## 3. Clean text and inspect the data

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0].copy()

df["label"].value_counts().plot(kind="bar", rot=0)
plt.title("Number of Messages by Class")
plt.xlabel("Class")
plt.ylabel("Number of messages")
plt.tight_layout()
plt.show()

print("Total messages:", len(df))
print("Spam proportion: {:.2%}".format(df["target"].mean()))

## 4. Split data and vectorize with TF-IDF
TF-IDF converts text into numerical features. We fit the vectorizer on the training data only to avoid data leakage.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"],
    df["target"],
    test_size=0.20,
    random_state=SEED,
    stratify=df["target"]
)

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train_text).toarray().astype("float32")
X_test = vectorizer.transform(X_test_text).toarray().astype("float32")

y_train = y_train.to_numpy().astype("float32")
y_test = y_test.to_numpy().astype("float32")

print("Training feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)

## 5. Build the neural network
The model uses dense layers, batch normalization, and dropout to help regularize training.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.40),
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.30),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 6. Train with early stopping
Early stopping monitors validation loss and restores the best model weights.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1
)

## 7. Plot training and validation curves

In [ ]:
history_dict = history.history
epochs = range(1, len(history_dict["loss"]) + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs, history_dict["accuracy"], marker="o", label="Training accuracy")
plt.plot(epochs, history_dict["val_accuracy"], marker="o", label="Validation accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs, history_dict["loss"], marker="o", label="Training loss")
plt.plot(epochs, history_dict["val_loss"], marker="o", label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Evaluate the model on unseen test data

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(
    y_test, predictions,
    labels=[0, 1],
    target_names=["Not Spam", "Spam"],
    zero_division=0
))

cm = confusion_matrix(y_test, predictions, labels=[0, 1])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Spam", "Spam"]).plot()
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 9. Predict new messages
Enter any new message in `sample_messages`. The model returns a class and its estimated spam probability.

In [ ]:
def predict_message(message):
    cleaned = clean_text(message)
    features = vectorizer.transform([cleaned]).toarray().astype("float32")
    spam_probability = float(model.predict(features, verbose=0)[0][0])
    predicted_label = "Spam" if spam_probability >= 0.5 else "Not Spam"
    confidence = spam_probability if predicted_label == "Spam" else 1 - spam_probability

    return {
        "message": message,
        "prediction": predicted_label,
        "confidence": round(confidence * 100, 2),
        "spam_probability": round(spam_probability * 100, 2)
    }

sample_messages = [
    "Congratulations! You have won a free prize. Click the link now!",
    "Hi, are we still meeting for lunch tomorrow?",
    "URGENT! Claim your cash reward now by calling this number.",
    "Please send me the notes from today's class."
]

results = pd.DataFrame([predict_message(msg) for msg in sample_messages])
display(results)

## 10. Save the trained model and vectorizer
The Keras model is saved as `spam_classifier.keras`. The TF-IDF vectorizer is saved separately because it is needed to convert future text into the same feature format.

In [ ]:
import pickle

model.save("spam_classifier.keras")
with open("tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

print("Saved: spam_classifier.keras")
print("Saved: tfidf_vectorizer.pkl")

## 11. Conclusion

In this project, we:
- Loaded and cleaned labeled SMS text.
- Converted messages into numeric TF-IDF features.
- Built a neural network using TensorFlow/Keras with dropout and batch normalization.
- Used early stopping to reduce overfitting.
- Plotted training/validation accuracy and loss.
- Evaluated the model using test accuracy, a classification report, and a confusion matrix.
- Tested the model on new messages and saved the model artifacts.

**Important:** Results depend on the dataset, train/test split, and training environment. Report the actual test metrics produced by your notebook; do not assume a fixed accuracy.